Description...

In [1]:
import sys
sys.path.append('/lhome/ific/c/ccortesp/Analysis/')

from libs import crudo

from datetime import datetime
import glob
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from pathlib import Path

%matplotlib inline
%load_ext autoreload
%autoreload 2

# Configuration

In [2]:
# --------------------------------
PROCESS_TYPE = 'bb0nu_hpr'                  # Options: 'radiogenics_hpr', 'radiogenics_lpr', 'bb2nu_hpr', 'bb2nu_lpr', 'bb0nu_hpr', 'bb0nu_lpr'
DATE = datetime.now().strftime('%d%m%Y')    # Options: today or some day (e.g '02122025')

INPUT_DIR = '/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/mc/'

# --------------------------------
# 2. ALPHA/ELECTRON DISCRIMINATION
# --------------------------------
SIZE_THRESHOLD = 2e3          # in [# of hits]
S1_ENERGY_THRESHOLD = 900     # in [PE]

# -------------------
# 3. DETECTOR REGIONS
# -------------------
# Geometric boundaries for event classification.
Z_LOW = 40          # in [mm]
Z_UP  = 1147        # in [mm]
R_UP  = 451.65      # in [mm]

# Merge

In [5]:
files_to_merge = glob.glob(os.path.join(INPUT_DIR, f"*{PROCESS_TYPE}*.h5"))
files_to_merge

['/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/mc/processed_mc_bb0nu_hpr_04052026.h5']

In [6]:
all_events_dfs = []
for file in files_to_merge:
    df = pd.read_hdf(file, key='Events')
    all_events_dfs.append(df)

MERGED_DF = pd.concat(all_events_dfs, ignore_index=True)
MERGED_DF

,event,npeak,E_peak_mev,X_bary,Y_bary,Z_bary,X_min,X_max,Y_min,Y_max,...,main_S1e,main_S1e_corr,main_S1w,main_S1h,main_S1t,main_S2e,main_S2w,main_S2h,main_S2t,main_S2q
0,0,0,2.287166,-334.389853,60.334637,580.350179,-435.975,-235.825,-16.925,137.575,...,771.305237,1037.296909,600.0,150.687195,10000.0,544024.562500,96.700,14874.986328,655485.06250,23381.218750
1,1,0,2.469598,-272.932612,-35.585210,746.288115,-373.775,-189.175,-140.325,75.375,...,942.674438,1153.685417,625.0,201.432327,10000.0,604717.187500,108.175,13597.195312,857492.06250,25743.246094
2,2,0,2.462551,-26.250674,-207.239493,373.267652,-81.325,57.625,-294.325,-124.275,...,708.525940,1044.374051,600.0,172.989059,10000.0,623079.750000,139.200,15805.226562,476488.78125,26589.275391
3,3,0,2.466902,91.976479,-316.800563,437.340221,-3.575,181.025,-386.625,-233.125,...,702.163635,1005.858697,600.0,158.754959,10000.0,598708.250000,120.375,12192.604492,530481.68750,25418.375000
4,4,0,2.323227,28.633914,245.315295,218.306734,-81.325,119.825,168.175,338.225,...,563.312866,949.537366,575.0,132.305420,10000.0,584060.812500,84.825,31454.078125,242485.96875,24582.085938
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1079455,999980,1,2.466397,-125.869796,-62.587416,478.901627,-204.725,-50.225,-139.825,29.225,...,725.239197,1039.455099,750.0,153.621338,10000.0,628257.500000,134.775,11394.843750,529484.62500,26686.722656
1079456,999981,0,2.410638,248.693212,-383.476428,146.126649,149.925,335.525,-479.925,-278.775,...,561.414062,993.256164,575.0,119.395927,10000.0,479950.531250,69.150,11654.717773,165492.34375,24349.796875
1079457,999982,0,0.414616,179.509189,-439.279243,538.530463,134.375,227.675,-479.925,-356.525,...,150.101044,204.567654,500.0,30.530266,10000.0,85818.679688,43.800,6174.355469,628492.75000,3877.101562
1079458,999983,0,2.465575,-258.489505,-145.324908,419.468323,-373.775,-173.625,-248.175,-78.125,...,696.243958,1041.351548,750.0,129.619156,10000.0,604461.875000,165.525,12759.539062,449494.90625,26487.246094


### Compute Global Event ID

We define a _global event ID_, 'cause we can have repeated _event ID_ across different isotopes and volumes
<br>
Additionally, we copy this information along isotope/volume tags into hits dataframe

In [7]:
# An original event is defined as a row in dataframe where at least one of the columns 
# ('event', 'Isotope', 'Volume') differs from the corresponding row below it (using `shift`).
event_OG = (MERGED_DF[['event', 'isotope', 'volume']] != MERGED_DF[['event', 'isotope', 'volume']].shift())

# If any column in event_OG is True, it means the row corresponds to the start of a new original event block.
new_event_block = event_OG.any(axis=1)

# Use `cumsum()` on the boolean mask to create a unique identifier for each contiguous block of hits 
# that belong to the same original event.
unique_block_id = new_event_block.cumsum()

# Assign a unique global event ID to each block of original events.
# The `factorize` function generates a unique integer code for each unique block ID.
MERGED_DF['global_event'] = pd.factorize(unique_block_id)[0]
print(f"{MERGED_DF['global_event'].nunique()} unique global events identified.")

956810 unique global events identified.


# Tagging Events

### By Particle

In [28]:
# particle_tagged_MERGED_DF = crudo.dm.tag_particles(MERGED_DF, size_threshold=SIZE_THRESHOLD, s1_energy_threshold=S1_ENERGY_THRESHOLD, event_column='global_event')
# particle_tagged_MERGED_DF

### By Detector Region

In [8]:
region_tagged_MERGED_DF = crudo.dm.tag_event_by_detector_region(MERGED_DF, z_cut_low=Z_LOW, z_cut_high=Z_UP, r_cut_high=R_UP, event_column='global_event')
region_tagged_MERGED_DF

,event,npeak,E_peak_mev,X_bary,Y_bary,Z_bary,X_min,X_max,Y_min,Y_max,...,main_S1w,main_S1h,main_S1t,main_S2e,main_S2w,main_S2h,main_S2t,main_S2q,global_event,region
0,0,0,2.287166,-334.389853,60.334637,580.350179,-435.975,-235.825,-16.925,137.575,...,600.0,150.687195,10000.0,544024.562500,96.700,14874.986328,655485.06250,23381.218750,0,fiducial
1,1,0,2.469598,-272.932612,-35.585210,746.288115,-373.775,-189.175,-140.325,75.375,...,625.0,201.432327,10000.0,604717.187500,108.175,13597.195312,857492.06250,25743.246094,1,fiducial
2,2,0,2.462551,-26.250674,-207.239493,373.267652,-81.325,57.625,-294.325,-124.275,...,600.0,172.989059,10000.0,623079.750000,139.200,15805.226562,476488.78125,26589.275391,2,fiducial
3,3,0,2.466902,91.976479,-316.800563,437.340221,-3.575,181.025,-386.625,-233.125,...,600.0,158.754959,10000.0,598708.250000,120.375,12192.604492,530481.68750,25418.375000,3,fiducial
4,4,0,2.323227,28.633914,245.315295,218.306734,-81.325,119.825,168.175,338.225,...,575.0,132.305420,10000.0,584060.812500,84.825,31454.078125,242485.96875,24582.085938,4,fiducial
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1079455,999980,1,2.466397,-125.869796,-62.587416,478.901627,-204.725,-50.225,-139.825,29.225,...,750.0,153.621338,10000.0,628257.500000,134.775,11394.843750,529484.62500,26686.722656,956805,fiducial
1079456,999981,0,2.410638,248.693212,-383.476428,146.126649,149.925,335.525,-479.925,-278.775,...,575.0,119.395927,10000.0,479950.531250,69.150,11654.717773,165492.34375,24349.796875,956806,tube
1079457,999982,0,0.414616,179.509189,-439.279243,538.530463,134.375,227.675,-479.925,-356.525,...,500.0,30.530266,10000.0,85818.679688,43.800,6174.355469,628492.75000,3877.101562,956807,tube
1079458,999983,0,2.465575,-258.489505,-145.324908,419.468323,-373.775,-173.625,-248.175,-78.125,...,750.0,129.619156,10000.0,604461.875000,165.525,12759.539062,449494.90625,26487.246094,956808,fiducial


# Output

In [36]:
# # H5 output filename
# MERGED_FILENAME = f'merged_tagged_runs_{DETECTOR_CONDITION}_{VERSION_TAG}.h5'
# MERGED_PATH = os.path.join('/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/', MERGED_FILENAME)
# print(f"\nSaving merged dataframe to: {MERGED_PATH}")

# H5 output filename
MERGED_FILENAME = f'merged_tagged_mc_{PROCESS_TYPE}_{DATE}.h5'
MERGED_PATH = os.path.join('/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/', MERGED_FILENAME)
print(f"\nSaving merged dataframe to: {MERGED_PATH}")


Saving merged dataframe to: /lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/merged_tagged_mc_bb0nu_hpr_04052026.h5


In [37]:
region_tagged_MERGED_DF.to_hdf(MERGED_PATH, key='Events', mode='w', format='table')
print('Done!')

Done!
